<a href="https://colab.research.google.com/github/KPR111/Google-Colab-Learning/blob/master/CV/Fixed_Learning_transfer_learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =======================================================
# FULLY SELF-CONTAINED SCRIPT FOR GOOGLE COLAB
# Downloads the Hymenoptera (Ants & Bees) dataset automatically,
# trains a ResNet18 via transfer learning from ImageNet,
# and logs everything to Weights & Biases.
# =======================================================

# ATTENTION - YOU NEED TO IMPLEMENT FOR FINE TUNING ALSO , WHICH IS EVEN GIVING BETTER RESULTS THAN THIS , IF PROPERLY IMPLEMENTD.

# 1. Install required packages
!pip install wandb -qqq

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
from torch.utils.data import DataLoader
import torchvision
from torchvision import datasets, transforms, models
import os
import copy
import wandb
from tqdm import tqdm
import urllib.request
import zipfile

# =======================================================
# 2. CONFIGURATION – adjust these parameters as needed
# =======================================================
BATCH_SIZE = 32
NUM_EPOCHS = 25
LEARNING_RATE = 0.001
MOMENTUM = 0.9
STEP_SIZE = 7
GAMMA = 0.1

# WandB project settings
wandb_project = "Fixed-Learning-transfer-learning"
wandb_run_name = "resnet18-imagenet-ants-bees"

# =======================================================
# 3. AUTOMATIC DATA DOWNLOAD & EXTRACTION
# =======================================================
print("Downloading Hymenoptera (Ants & Bees) dataset...")
url = "https://download.pytorch.org/tutorial/hymenoptera_data.zip"
zip_path = "hymenoptera_data.zip"
extract_path = "hymenoptera_data"

# Download the dataset
urllib.request.urlretrieve(url, zip_path)
print("Download complete!")

# Extract the dataset
print("Extracting dataset...")
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)
print("Extraction complete!")

# Paths to train and validation folders
data_dir = os.path.join(extract_path, "hymenoptera_data")
train_dir = os.path.join(data_dir, "train")
val_dir = os.path.join(data_dir, "val")

# =======================================================
# 4. DATA TRANSFORMS (Resize to 224x224 and normalize)
# =======================================================
# ImageNet normalization values
mean = [0.485, 0.456, 0.406]
std  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

# Create datasets and dataloaders
train_dataset = datasets.ImageFolder(train_dir, transform=train_transform)
val_dataset = datasets.ImageFolder(val_dir, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

class_names = train_dataset.classes
print(f"Classes: {class_names}")
print(f"Train samples: {len(train_dataset)}, Validation samples: {len(val_dataset)}")

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# =======================================================
# 5. WANDB INITIALIZATION
# =======================================================
wandb.init(project=wandb_project, name=wandb_run_name, config={
    "num_classes": len(class_names),
    "batch_size": BATCH_SIZE,
    "num_epochs": NUM_EPOCHS,
    "learning_rate": LEARNING_RATE,
    "momentum": MOMENTUM,
    "step_size": STEP_SIZE,
    "gamma": GAMMA,
    "pretrained": "ImageNet_ResNet18",
    "dataset": "Hymenoptera_Ants_Bees"
})

# =======================================================
# 6. LOAD PRETRAINED MODEL AND REPLACE CLASSIFIER
# =======================================================
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

# Freeze all layers (keep ImageNet features)
for param in model.parameters():
    param.requires_grad = False

# Replace classifier head (only this will be trained)
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, len(class_names))
model = model.to(device)

# Only train the new head
optimizer = optim.SGD(model.fc.parameters(), lr=LEARNING_RATE, momentum=MOMENTUM)
scheduler = lr_scheduler.StepLR(optimizer, step_size=STEP_SIZE, gamma=GAMMA)
criterion = nn.CrossEntropyLoss()

# Watch model with wandb
wandb.watch(model, log="all", log_freq=10)

# =======================================================
# 7. SEPARATE TRAIN / VALIDATION FUNCTIONS
# =======================================================
def train_one_epoch(model, dataloader, criterion, optimizer, epoch):
    model.train()
    running_loss = 0.0
    running_corrects = 0

    loop = tqdm(dataloader, desc=f"Train Epoch {epoch}", leave=False)
    for inputs, labels in loop:
        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        _, preds = torch.max(outputs, 1)
        running_loss += loss.item() * inputs.size(0)
        running_corrects += torch.sum(preds == labels.data)

        loop.set_postfix(loss=loss.item())

    epoch_loss = running_loss / len(dataloader.dataset)
    epoch_acc = running_corrects.double() / len(dataloader.dataset)
    return epoch_loss, epoch_acc


def validate_one_epoch(model, dataloader, criterion, epoch):
    model.eval()
    running_loss = 0.0
    running_corrects = 0

    with torch.no_grad():
        loop = tqdm(dataloader, desc=f"Val Epoch {epoch}", leave=False)
        for inputs, labels in loop:
            inputs = inputs.to(device)
            labels = labels.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, labels)

            _, preds = torch.max(outputs, 1)
            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)

            loop.set_postfix(loss=loss.item())

    epoch_loss = running_loss / len(dataloader.dataset)
    epoch_acc = running_corrects.double() / len(dataloader.dataset)
    return epoch_loss, epoch_acc

# =======================================================
# 8. MAIN TRAINING LOOP
# =======================================================
best_model_wts = copy.deepcopy(model.state_dict())
best_acc = 0.0

for epoch in range(1, NUM_EPOCHS + 1):
    print(f"\nEpoch {epoch}/{NUM_EPOCHS}")
    print("-" * 30)

    # Training phase
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, epoch)

    # Validation phase
    val_loss, val_acc = validate_one_epoch(model, val_loader, criterion, epoch)

    # Step the scheduler
    scheduler.step()

    # Log to wandb
    wandb.log({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_accuracy": train_acc,
        "val_loss": val_loss,
        "val_accuracy": val_acc,
    })

    print(f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f}")
    print(f"Val   Loss: {val_loss:.4f} Acc: {val_acc:.4f}")

    # Save best model based on validation accuracy
    if val_acc > best_acc:
        best_acc = val_acc
        best_model_wts = copy.deepcopy(model.state_dict())
        print(f"--> New best model! Val Acc: {best_acc:.4f}")

print(f"\nBest validation accuracy: {best_acc:.4f}")
model.load_state_dict(best_model_wts)

# =======================================================
# 9. SAVE MODEL AND FINISH WANDB RUN
# =======================================================
torch.save(model.state_dict(), "best_model.pth")
wandb.save("best_model.pth")
print("Model saved as 'best_model.pth' and uploaded to wandb.")

wandb.finish()
print("\nTraining complete. Check your Weights & Biases dashboard for loss/accuracy graphs.")

Download complete!
Extracting dataset...
Extraction complete!
Classes: ['ants', 'bees']
Train samples: 244, Validation samples: 153
Using device: cuda:0


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


...

Train Loss: 0.2113 Acc: 0.9467
Val   Loss: 0.1857 Acc: 0.9412

Epoch 25/25
------------------------------


wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.


Train Loss: 0.2137 Acc: 0.9426
Val   Loss: 0.1839 Acc: 0.9477

Best validation accuracy: 0.9477
Model saved as 'best_model.pth' and uploaded to wandb.


epoch,▁▁▂▂▂▂▃▃▃▄▄▄▅▅▅▅▆▆▆▇▇▇▇██
train_accuracy,▁▄▆▇▇▇▇█▇████▇████▇█▇████
train_loss,█▆▄▃▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_accuracy,▁▆▇██████████████████████
val_loss,█▅▃▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,25
train_accuracy,0.94262
train_loss,0.21365
val_accuracy,0.94771
val_loss,0.18394



Training complete. Check your Weights & Biases dashboard for loss/accuracy graphs.
